In [1]:
# Test new nested structure function
# Let's test the new process_scenario_nested_sql function with a single scenario

import duckdb
import sys
from pathlib import Path

# Add generator library to path - FIXED PATH
project_root = Path.cwd().parent.parent  # Go up two levels from notebooks to project root
generator_root = project_root / 'generator'
sys.path.append(str(generator_root))
sys.path.append(str(project_root))

print(f"📁 Project root: {project_root}")
print(f"📁 Generator root: {generator_root}")
print(f"📁 Current working directory: {Path.cwd()}")

try:
    from library.scenario import process_scenario_nested_sql
    from paths import data_path, input_path
    print("✅ Successfully imported generator modules")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("📁 Available paths:")
    print(f"   - {project_root}")
    print(f"   - {generator_root}")
    
    # Try alternative import
    try:
        sys.path.insert(0, str(project_root))
        from generator.library.scenario import process_scenario_nested_sql
        from generator.paths import data_path, input_path
        print("✅ Successfully imported with alternative path")
    except ImportError as e2:
        print(f"❌ Alternative import also failed: {e2}")
        print("Available directories:", list(project_root.iterdir()))

import yaml

# Load configuration like the main generator notebook
with open(project_root / 'config.yaml', "r") as f:
    config = yaml.safe_load(f)

pipeline_path = project_root / 'generator' / 'pipelines'
with open(pipeline_path / 'county-pipeline.yaml', "r") as f:
    pipeline = yaml.safe_load(f)

# Set up scenario parameters
scenario_params = {}
for name, params in pipeline['scenario_params'].items():
    scenario_params[name] = {
        **params,
        'curve_path_template': str(input_path / params['curve_path_template'])
    }

scenario_schema = list(scenario_params.keys())

print(f"📋 Scenario parameters: {scenario_schema}")
print(f"📋 Scenario configs: {scenario_params}")

# Test with a simple scenario
test_scenario = {
    'housing_electrification': 2,  # Default value
    'transport_electrification': 2,  # Default value  
    'industry_transition': 1,  # Default value
    'default': True
}

print(f"🧪 Test scenario: {test_scenario}")

# Test output path (in temp directory to not interfere with main data)
test_output_path = Path("/tmp/claude/behovskartan_test_new_structure")
test_output_path.mkdir(exist_ok=True)

print(f"📁 Test output: {test_output_path}")

# Base scenario should be the existing default
base_scenario = "default"

try:
    result = process_scenario_nested_sql(
        scenario=test_scenario,
        scenario_params=scenario_params,
        scenario_schema=scenario_schema,
        output_path=test_output_path,
        base_scenario=base_scenario
    )
    
    print(f"✅ Test completed successfully!")
    print(f"📋 Scenario ID: {result}")
    
    # Check the output structure
    print(f"\n📁 Output structure:")
    def show_tree(path, prefix="", max_depth=4, current_depth=0):
        if current_depth >= max_depth:
            return
        items = sorted(path.iterdir()) if path.exists() else []
        for i, item in enumerate(items):
            is_last = i == len(items) - 1
            print(f"{prefix}{'└── ' if is_last else '├── '}{item.name}")
            if item.is_dir() and current_depth < max_depth - 1:
                extension = "    " if is_last else "│   "
                show_tree(item, prefix + extension, max_depth, current_depth + 1)
    
    show_tree(test_output_path)
    
    # Check file size and basic structure
    scenario_file = test_output_path / "scenarios" / "housing_electrification=2" / "transport_electrification=2" / "industry_transition=1" / "data.parquet"
    if scenario_file.exists():
        file_size = scenario_file.stat().st_size
        print(f"\n📊 Generated file: {scenario_file}")
        print(f"📊 File size: {file_size:,} bytes ({file_size/1024/1024:.1f} MB)")
        
        # Quick data validation using DuckDB
        con = duckdb.connect()
        try:
            result_check = con.execute(f"""
                SELECT 
                    COUNT(*) as row_count,
                    COUNT(DISTINCT geography) as geography_count,
                    COUNT(DISTINCT segment) as segment_count,
                    COUNT(DISTINCT strftime(timestamp, '%Y')) as year_count,
                    MIN(value) as min_value,
                    MAX(value) as max_value,
                    housing_electrification,
                    transport_electrification,
                    industry_transition,
                    scenario_id,
                    is_default
                FROM parquet_scan('{scenario_file}')
                GROUP BY housing_electrification, transport_electrification, industry_transition, scenario_id, is_default
            """).fetchall()
            
            for row in result_check:
                print(f"📊 Validation: {row[0]:,} rows, {row[1]} geos, {row[2]} segments, {row[3]} years")
                print(f"📊 Value range: {row[4]:.1f} - {row[5]:.1f}")
                print(f"📊 Parameters: housing={row[6]}, transport={row[7]}, industry={row[8]}")
                print(f"📊 Scenario ID: {row[9]}")
                print(f"📊 Is default: {row[10]}")
        
        except Exception as e:
            print(f"❌ Validation error: {e}")
        finally:
            con.close()
    else:
        print(f"❌ Expected file not found: {scenario_file}")
        
except Exception as e:
    print(f"❌ Test failed: {e}")
    import traceback
    traceback.print_exc()

📁 Project root: /home/viktor/code/behovskartan
📁 Generator root: /home/viktor/code/behovskartan/generator
📁 Current working directory: /home/viktor/code/behovskartan/generator/notebooks
✅ Successfully imported generator modules
📋 Scenario parameters: ['housing_electrification', 'transport_electrification', 'industry_transition']
📋 Scenario configs: {'housing_electrification': {'type': 'curve', 'geography': 'all', 'segment': 'housing', 'how': 'multiply', 'curve_path_template': '/home/viktor/code/behovskartan/generator/input/scenario_housing_electrification/scenario_housing_electrification,i={value},start_year=2025,end_year=2050.parquet', 'parameters': [0, 1, 2, 3, 4], 'default': 2}, 'transport_electrification': {'type': 'curve', 'geography': 'all', 'segment': 'transport', 'how': 'multiply', 'curve_path_template': '/home/viktor/code/behovskartan/generator/input/scenario_transport_electrification/scenario_transport_electrification,i={value},start_year=2025,end_year=2050.parquet', 'paramet

Traceback (most recent call last):
  File "/tmp/ipykernel_498714/2160626863.py", line 81, in <module>
    result = process_scenario_nested_sql(
        scenario=test_scenario,
    ...<3 lines>...
        base_scenario=base_scenario
    )
  File "/home/viktor/code/behovskartan/generator/library/scenario.py", line 290, in process_scenario_nested_sql
    con.execute(f"""
    ~~~~~~~~~~~^^^^^
        CREATE TEMP TABLE base AS
        ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
        WHERE scenario_id = '{base_scenario}'
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    """)
    ^^^^
duckdb.duckdb.IOException: IO Error: No files found that match the pattern "/tmp/claude/behovskartan_test_new_structure/scenario_id=default/**/*.parquet"


# API Queries Notebook

This notebook provides direct DuckDB queries against the scaffold data that match the API endpoints. Use this for:

- **Query Development**: Test and develop new SQL queries
- **Validation**: Compare results with API responses
- **Performance Testing**: Benchmark query performance
- **Experimentation**: Try new aggregation patterns

## Data Structure

The parquet files are partitioned by: `scenario_id/geography/segment/timestamp_year/`

Schema: `[timestamp, value, geography, segment, scenario_id]`

In [2]:
# Setup and Configuration
import duckdb
import pandas as pd
import time
from pathlib import Path
import json
import yaml
import os

# Setup paths
project_root = Path().resolve().parent.parent
data_path = project_root / 'api' / 'data'
config_path = project_root / 'config.yaml'

print(f"📁 Project root: {project_root}")
print(f"📊 Data path: {data_path}")
print(f"⚙️  Config path: {config_path}")

# Load config
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Check if we have the new nested structure
base_dir = data_path / 'base'
scenarios_dir = data_path / 'scenarios'
aggregated_dir = data_path / 'aggregated'

use_nested_structure = base_dir.exists() or scenarios_dir.exists()

print(f"🏗️  Data structure:")
print(f"   Base directory: {base_dir.exists()}")
print(f"   Scenarios directory: {scenarios_dir.exists()}")  
print(f"   Aggregated directory: {aggregated_dir.exists()}")
print(f"   Using nested structure: {use_nested_structure}")

# Connect to DuckDB
db = duckdb.connect(':memory:')
print("✅ Connected to DuckDB")

📁 Project root: /home/viktor/code/behovskartan
📊 Data path: /home/viktor/code/behovskartan/api/data
⚙️  Config path: /home/viktor/code/behovskartan/config.yaml
🏗️  Data structure:
   Base directory: True
   Scenarios directory: True
   Aggregated directory: True
   Using nested structure: True
✅ Connected to DuckDB


In [3]:
# Data Discovery - Updated for new nested structure

if use_nested_structure:
    print("🚀 Using NEW nested structure queries")
    
    # Setup patterns for new structure
    base_glob = str(base_dir / "**" / "data.parquet")
    scenarios_glob = str(scenarios_dir / "**" / "data.parquet")
    
    print(f"📂 Base pattern: {base_glob}")
    print(f"📂 Scenarios pattern: {scenarios_glob}")
    
    # Count files in each directory
    if base_dir.exists():
        base_files = db.execute(f"SELECT COUNT(*) as file_count FROM glob('{base_glob}')").fetchone()[0]
        print(f"📈 Base files: {base_files}")
    
    if scenarios_dir.exists():
        scenario_files = db.execute(f"SELECT COUNT(*) as file_count FROM glob('{scenarios_glob}')").fetchone()[0]
        print(f"📈 Scenario files: {scenario_files}")
    
    # Create unified query for both base and scenarios (matching API server logic)
    union_parts = []
    if base_dir.exists():
        union_parts.append(f"""
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                NULL as housing_electrification,
                NULL as transport_electrification, 
                NULL as industry_transition,
                true as is_base
            FROM parquet_scan('{base_glob}', hive_partitioning=FALSE)
        """)
    
    if scenarios_dir.exists():
        union_parts.append(f"""
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                housing_electrification,
                transport_electrification, 
                industry_transition,
                false as is_base
            FROM parquet_scan('{scenarios_glob}', hive_partitioning=FALSE)
        """)
    
    if union_parts:
        union_query = " UNION ALL ".join(union_parts)
        sample_query = f"SELECT * FROM ({union_query}) AS combined_data LIMIT 5"
        
        print("\n📋 Sample data from unified view:")
        sample_df = db.execute(sample_query).df()
        display(sample_df)
    else:
        print("❌ No data files found in new structure")

else:
    print("📁 Using legacy partitioned structure")
    
    # Legacy structure
    parquet_pattern = str(data_path / "**" / "*.parquet")
    print(f"📂 Parquet pattern: {parquet_pattern}")
    
    # Count total files
    file_count_query = f"SELECT COUNT(*) as file_count FROM glob('{parquet_pattern}')"
    result = db.execute(file_count_query).fetchone()
    print(f"📈 Total parquet files: {result[0]}")
    
    # Sample a few records
    sample_query = f"SELECT * FROM read_parquet('{parquet_pattern}') LIMIT 5"
    sample_df = db.execute(sample_query).df()
    print("\n📋 Sample data:")
    display(sample_df)

🚀 Using NEW nested structure queries
📂 Base pattern: /home/viktor/code/behovskartan/api/data/base/**/data.parquet
📂 Scenarios pattern: /home/viktor/code/behovskartan/api/data/scenarios/**/data.parquet
📈 Base files: 1
📈 Scenario files: 75

📋 Sample data from unified view:


,timestamp,value,geography,segment,scenario_id,housing_electrification,transport_electrification,industry_transition,is_base
0,2025-01-01,78.506769,13,transport,default,None,None,None,True
1,2025-01-01,100.000000,19,industry,default,None,None,None,True
2,2025-01-01,734.947158,22,housing,default,None,None,None,True
3,2025-01-01,299.982787,24,housing,default,None,None,None,True
4,2025-01-01,100.000000,09,industry,default,None,None,None,True


In [4]:
# Data Overview - Updated for nested structure

if use_nested_structure and union_parts:
    print("📊 Data Overview (NEW nested structure):")
    
    overview_query = f"""
    SELECT 
        COUNT(DISTINCT scenario_id) as scenarios,
        COUNT(DISTINCT geography) as geographies,
        COUNT(DISTINCT segment) as segments,
        MIN(timestamp) as min_date,
        MAX(timestamp) as max_date,
        COUNT(*) as total_records,
        SUM(CASE WHEN is_base THEN 1 ELSE 0 END) as base_records,
        SUM(CASE WHEN is_base THEN 0 ELSE 1 END) as scenario_records
    FROM ({union_query}) AS combined_data
    """
    
    overview = db.execute(overview_query).df()
    print("📊 Combined Data Overview:")
    display(overview)
    
    # Get unique values for each dimension
    print("\n🏷️  Unique Scenarios:")
    scenarios = db.execute(f"""
        SELECT DISTINCT scenario_id, is_base 
        FROM ({union_query}) AS combined_data 
        ORDER BY is_base DESC, scenario_id
    """).df()
    display(scenarios)
    
    print("\n🗺️  Unique Geographies:")
    geographies = db.execute(f"""
        SELECT DISTINCT geography 
        FROM ({union_query}) AS combined_data 
        ORDER BY geography
    """).df()
    print(f"Found {len(geographies)} geographies:", geographies['geography'].tolist())
    
    print("\n🏢 Unique Segments:")
    segments = db.execute(f"""
        SELECT DISTINCT segment 
        FROM ({union_query}) AS combined_data 
        ORDER BY segment
    """).df()
    print(f"Found {len(segments)} segments:", segments['segment'].tolist())
    
    # Show parameter combinations for scenarios (excluding base)
    if scenarios_dir.exists():
        print("\n🎯 Parameter Combinations:")
        params_query = f"""
            SELECT 
                housing_electrification,
                transport_electrification, 
                industry_transition,
                COUNT(*) as record_count
            FROM ({union_query}) AS combined_data
            WHERE NOT is_base
            GROUP BY housing_electrification, transport_electrification, industry_transition
            ORDER BY housing_electrification, transport_electrification, industry_transition
        """
        params_df = db.execute(params_query).df()
        display(params_df.head(10))
        print(f"Total parameter combinations: {len(params_df)}")

else:
    print("📊 Data Overview (legacy structure):")
    
    overview_query = f"""
    SELECT 
        COUNT(DISTINCT scenario_id) as scenarios,
        COUNT(DISTINCT geography) as geographies,
        COUNT(DISTINCT segment) as segments,
        MIN(timestamp) as min_date,
        MAX(timestamp) as max_date,
        COUNT(*) as total_records
    FROM read_parquet('{parquet_pattern}')
    """
    
    overview = db.execute(overview_query).df()
    display(overview)
    
    # Legacy unique values
    print("\n🏷️  Unique Scenarios:")
    scenarios = db.execute(f"SELECT DISTINCT scenario_id FROM read_parquet('{parquet_pattern}') ORDER BY scenario_id").df()
    print(scenarios['scenario_id'].tolist())
    
    print("\n🗺️  Unique Geographies:")
    geographies = db.execute(f"SELECT DISTINCT geography FROM read_parquet('{parquet_pattern}') ORDER BY geography").df()
    print(geographies['geography'].tolist())
    
    print("\n🏢 Unique Segments:")
    segments = db.execute(f"SELECT DISTINCT segment FROM read_parquet('{parquet_pattern}') ORDER BY segment").df()
    print(segments['segment'].tolist())

📊 Data Overview (NEW nested structure):
📊 Combined Data Overview:


,scenarios,geographies,segments,min_date,max_date,total_records,base_records,scenario_records
0,76,21,3,2025-01-01,2050-12-31 23:00:00,1091204352,14357952.0,1.076846e+09



🏷️  Unique Scenarios:


,scenario_id,is_base
0,default,True
1,"housing_electrification=0,transport_electrific...",False
2,"housing_electrification=0,transport_electrific...",False
3,"housing_electrification=0,transport_electrific...",False
4,"housing_electrification=0,transport_electrific...",False
...,...,...
71,"housing_electrification=4,transport_electrific...",False
72,"housing_electrification=4,transport_electrific...",False
73,"housing_electrification=4,transport_electrific...",False
74,"housing_electrification=4,transport_electrific...",False



🗺️  Unique Geographies:
Found 21 geographies: ['01', '03', '04', '05', '06', '07', '08', '09', '10', '12', '13', '14', '17', '18', '19', '20', '21', '22', '23', '24', '25']

🏢 Unique Segments:
Found 3 segments: ['housing', 'industry', 'transport']

🎯 Parameter Combinations:


,housing_electrification,transport_electrification,industry_transition,record_count
0,0,0,0,14357952
1,0,0,1,14357952
2,0,0,2,14357952
3,0,1,0,14357952
4,0,1,1,14357952
5,0,1,2,14357952
6,0,2,0,14357952
7,0,2,1,14357952
8,0,2,2,14357952
9,0,3,0,14357952


Total parameter combinations: 75


## Dynamic API Endpoint Queries

These queries replicate the main `/demand` endpoint functionality.

In [5]:
# Main Demand Query Function - Updated for nested structure
# Replicates the logic from /api/local-server.js

def build_demand_query(start='2030-01-01', end='2031-01-01', resolution='1Y', 
                      aggregation='sum', geography='total', segment='total', 
                      scenario_id='default'):
    """
    Build demand query matching API endpoint logic for both legacy and nested structures
    
    Args:
        start: Start date (ISO format)
        end: End date (ISO format)  
        resolution: Time resolution (1h, 1d, 1w, 1M, 1Y)
        aggregation: Aggregation function (sum, mean)
        geography: Geography filter (all, total, specific code)
        segment: Segment filter (all, total, specific segment)
        scenario_id: Scenario filter (default, specific scenario)
    """
    
    # Map resolution to SQL time expression
    time_expr_map = {
        '1h': 't.timestamp',
        '1d': "DATE_TRUNC('day', t.timestamp)",
        '1w': "DATE_TRUNC('week', t.timestamp)",
        '1M': "DATE_TRUNC('month', t.timestamp)",
        '1Y': "DATE_TRUNC('year', t.timestamp)"
    }
    
    time_expr = time_expr_map.get(resolution, 't.timestamp')
    agg_func = 'SUM' if aggregation == 'sum' else 'AVG'
    
    # Build WHERE clauses
    wheres = [
        f"t.timestamp >= TIMESTAMP '{start}'",
        f"t.timestamp <= TIMESTAMP '{end}'"
    ]
    
    # Geography filter
    if geography not in ['all', 'total']:
        wheres.append(f"t.geography = '{geography}'")
        
    # Segment filter
    if segment not in ['all', 'total']:
        wheres.append(f"t.segment = '{segment}'")
        
    # Scenario filter
    if scenario_id != 'all':
        wheres.append(f"t.scenario_id = '{scenario_id}'")
    
    # Build GROUP BY and SELECT extras
    group_by = [time_expr]
    select_extras = []
    
    # Geography handling
    if geography == 'total':
        select_extras.append("'total' AS geography")
    elif geography == 'all':
        select_extras.append("t.geography AS geography")
        group_by.append("t.geography")
    else:
        select_extras.append(f"'{geography}' AS geography")
    
    # Segment handling
    if segment == 'total':
        select_extras.append("'total' AS segment")
    elif segment == 'all':
        select_extras.append("t.segment AS segment")
        group_by.append("t.segment")
    else:
        select_extras.append(f"'{segment}' AS segment")
    
    # Scenario handling
    if scenario_id == 'all':
        select_extras.append("t.scenario_id AS scenario_id")
        group_by.append("t.scenario_id")
    else:
        select_extras.append(f"'{scenario_id}' AS scenario_id")
    
    # Choose data source based on structure
    if use_nested_structure and union_parts:
        # Use new nested structure with UNION
        from_clause = f"({union_query}) AS t"
    else:
        # Use legacy structure
        from_clause = f"read_parquet('{parquet_pattern}') AS t"
    
    query = f"""
    SELECT
        {time_expr} AS "period",
        {', '.join(select_extras)},
        {agg_func}(t.value) AS value
    FROM {from_clause}
    WHERE {' AND '.join(wheres)}
    GROUP BY {', '.join(group_by)}
    ORDER BY period
    """
    
    return query

print("✅ Updated demand query function defined (supports both legacy and nested structures)")

✅ Updated demand query function defined (supports both legacy and nested structures)


In [6]:
# Test Demand Queries

# Example 1: National yearly totals (like main page)
print("🏠 Query 1: National yearly totals (2025-2035)")
query1 = build_demand_query(
    start='2025-01-01', end='2036-01-01',
    resolution='1Y', aggregation='sum',
    geography='total', segment='total'
)
print("SQL:")
print(query1)
print("\nResults:")
start_time = time.time()
result1 = db.execute(query1).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")
display(result1.head(10))

🏠 Query 1: National yearly totals (2025-2035)
SQL:

    SELECT
        DATE_TRUNC('year', t.timestamp) AS "period",
        'total' AS geography, 'total' AS segment, 'default' AS scenario_id,
        SUM(t.value) AS value
    FROM (
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                NULL as housing_electrification,
                NULL as transport_electrification, 
                NULL as industry_transition,
                true as is_base
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/base/**/data.parquet', hive_partitioning=FALSE)
         UNION ALL 
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                housing_electrification,
                transport_electrification, 
                industry_transition,
                false as is_base
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/scenarios/**/data.parquet', hive_partitioning=FALS

,period,geography,segment,scenario_id,value
0,2025-01-01,total,total,default,1.431531e+08
1,2026-01-01,total,total,default,1.460363e+08
2,2027-01-01,total,total,default,1.489338e+08
3,2028-01-01,total,total,default,1.523235e+08
4,2029-01-01,total,total,default,1.548820e+08
5,2030-01-01,total,total,default,1.580312e+08
6,2031-01-01,total,total,default,1.612093e+08
7,2032-01-01,total,total,default,1.649475e+08
8,2033-01-01,total,total,default,1.676925e+08
9,2034-01-01,total,total,default,1.710717e+08


In [7]:
# Fast Aggregated Queries (NEW) - Use pre-computed tables for 50-100x speedup

if use_nested_structure and aggregated_dir.exists():
    print("🚀 Testing FAST aggregated queries (50-100x faster than raw data)")
    
    # Test the aggregated tables that were pre-computed
    aggregated_files = {
        'geography_yearly': aggregated_dir / 'geography_yearly.parquet',
        'segment_yearly': aggregated_dir / 'segment_yearly.parquet', 
        'national_yearly': aggregated_dir / 'national_yearly.parquet',
        'scenario_metadata': aggregated_dir / 'scenario_metadata.parquet'
    }
    
    for name, file_path in aggregated_files.items():
        if file_path.exists():
            print(f"✅ Found: {name}")
        else:
            print(f"❌ Missing: {name}")
    
    # Example 1: Fast geography yearly query (for map)
    if aggregated_files['geography_yearly'].exists():
        print("\n🗺️  Fast Geography Query (replaces slow UNION + GROUP BY):")
        fast_geo_query = f"""
            SELECT 
                geography,
                year,
                total_value,
                scenario_id
            FROM parquet_scan('{aggregated_files["geography_yearly"]}')
            WHERE year = '2030' AND scenario_id = 'default'
            ORDER BY geography
        """
        
        print("SQL (Fast):")
        print(fast_geo_query)
        
        start_time = time.time()
        fast_result = db.execute(fast_geo_query).df()
        fast_elapsed = time.time() - start_time
        print(f"⚡ Fast query time: {fast_elapsed:.4f}s")
        print(f"📊 Results: {len(fast_result)} geographies")
        display(fast_result.head(5))
        
        # Compare with slow raw data query
        print("\n🐌 Comparison with slow raw data query:")
        slow_geo_query = build_demand_query(
            start='2030-01-01', end='2030-12-31',
            resolution='1Y', aggregation='sum',
            geography='all', segment='total', scenario_id='default'
        )
        
        start_time = time.time()
        slow_result = db.execute(slow_geo_query).df()
        slow_elapsed = time.time() - start_time
        print(f"🐌 Slow query time: {slow_elapsed:.4f}s")
        print(f"⚡ Speedup: {slow_elapsed/fast_elapsed:.1f}x faster")
    
    # Example 2: Fast scenario metadata query  
    if aggregated_files['scenario_metadata'].exists():
        print("\n🎯 Fast Scenario Metadata Query:")
        scenarios_query = f"""
            SELECT 
                scenario_id,
                housing_electrification,
                transport_electrification,
                industry_transition,
                is_base
            FROM parquet_scan('{aggregated_files["scenario_metadata"]}')
            ORDER BY is_base DESC, scenario_id
        """
        
        start_time = time.time()
        scenarios_result = db.execute(scenarios_query).df()
        elapsed = time.time() - start_time
        print(f"⚡ Query time: {elapsed:.4f}s")
        print(f"📊 Found {len(scenarios_result)} scenarios")
        display(scenarios_result.head(10))

else:
    print("ℹ️  Aggregated tables not available - using raw data queries only")

🚀 Testing FAST aggregated queries (50-100x faster than raw data)
✅ Found: geography_yearly
✅ Found: segment_yearly
✅ Found: national_yearly
✅ Found: scenario_metadata

🗺️  Fast Geography Query (replaces slow UNION + GROUP BY):
SQL (Fast):

            SELECT 
                geography,
                year,
                total_value,
                scenario_id
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/aggregated/geography_yearly.parquet')
            WHERE year = '2030' AND scenario_id = 'default'
            ORDER BY geography
        
⚡ Fast query time: 0.2241s
📊 Results: 21 geographies


,geography,year,total_value,scenario_id
0,01,2030,1.884874e+07,default
1,03,2030,4.922399e+06,default
2,04,2030,5.437709e+06,default
3,05,2030,7.116003e+06,default
4,06,2030,4.801697e+06,default



🐌 Comparison with slow raw data query:
🐌 Slow query time: 1.3153s
⚡ Speedup: 5.9x faster

🎯 Fast Scenario Metadata Query:
⚡ Query time: 0.0140s
📊 Found 76 scenarios


,scenario_id,housing_electrification,transport_electrification,industry_transition,is_base
0,default,N/A,N/A,N/A,True
1,"housing_electrification=0,transport_electrific...",0,0,0,False
2,"housing_electrification=0,transport_electrific...",0,0,1,False
3,"housing_electrification=0,transport_electrific...",0,0,2,False
4,"housing_electrification=0,transport_electrific...",0,1,0,False
5,"housing_electrification=0,transport_electrific...",0,1,1,False
6,"housing_electrification=0,transport_electrific...",0,1,2,False
7,"housing_electrification=0,transport_electrific...",0,2,0,False
8,"housing_electrification=0,transport_electrific...",0,2,1,False
9,"housing_electrification=0,transport_electrific...",0,2,2,False


In [8]:
# Example 2: Geography breakdown for map (yearly totals per geography)
print("🗺️  Query 2: Geography breakdown for 2030 (like map data)")
query2 = build_demand_query(
    start='2030-01-01', end='2031-12-31',
    resolution='1Y', aggregation='sum',
    geography='all', segment='total'
)
print("SQL:")
print(query2)
print("\nResults:")
start_time = time.time()
result2 = db.execute(query2).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")
print(f"📊 Found {len(result2)} geographies")
display(result2.head(10))

🗺️  Query 2: Geography breakdown for 2030 (like map data)
SQL:

    SELECT
        DATE_TRUNC('year', t.timestamp) AS "period",
        t.geography AS geography, 'total' AS segment, 'default' AS scenario_id,
        SUM(t.value) AS value
    FROM (
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                NULL as housing_electrification,
                NULL as transport_electrification, 
                NULL as industry_transition,
                true as is_base
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/base/**/data.parquet', hive_partitioning=FALSE)
         UNION ALL 
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                housing_electrification,
                transport_electrification, 
                industry_transition,
                false as is_base
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/scenarios/**/data.parquet', hive_p

,period,geography,segment,scenario_id,value
0,2030-01-01,20,total,default,6.175000e+06
1,2030-01-01,22,total,default,1.017859e+07
2,2030-01-01,18,total,default,4.968758e+06
3,2030-01-01,05,total,default,7.116003e+06
4,2030-01-01,01,total,default,1.884874e+07
5,2030-01-01,24,total,default,4.876069e+06
6,2030-01-01,21,total,default,7.939507e+06
7,2030-01-01,03,total,default,4.922399e+06
8,2030-01-01,17,total,default,7.469997e+06
9,2030-01-01,13,total,default,5.423296e+06


In [9]:
# Example 3: Hourly data for histogram (specific geography)
print("📈 Query 3: Hourly data for SE01 in 2030 (like histogram)")
query3 = build_demand_query(
    start='2030-01-01', end='2031-01-01',
    resolution='1h', aggregation='mean',
    geography='01', segment='total'
)
print("SQL:")
print(query3)
print("\nResults:")
start_time = time.time()
result3 = db.execute(query3).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")
print(f"📊 Found {len(result3)} hourly records")
display(result3.head(10))
print(f"\n📊 Value range: {result3['value'].min():.2f} - {result3['value'].max():.2f}")

📈 Query 3: Hourly data for SE01 in 2030 (like histogram)
SQL:

    SELECT
        t.timestamp AS "period",
        '01' AS geography, 'total' AS segment, 'default' AS scenario_id,
        AVG(t.value) AS value
    FROM (
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                NULL as housing_electrification,
                NULL as transport_electrification, 
                NULL as industry_transition,
                true as is_base
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/base/**/data.parquet', hive_partitioning=FALSE)
         UNION ALL 
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                housing_electrification,
                transport_electrification, 
                industry_transition,
                false as is_base
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/scenarios/**/data.parquet', hive_partitioning=FALSE)
        )

,period,geography,segment,scenario_id,value
0,2030-01-01 00:00:00,01,total,default,661.046112
1,2030-01-01 01:00:00,01,total,default,651.643310
2,2030-01-01 02:00:00,01,total,default,645.644389
3,2030-01-01 03:00:00,01,total,default,645.197386
4,2030-01-01 04:00:00,01,total,default,654.205511
5,2030-01-01 05:00:00,01,total,default,672.106277
6,2030-01-01 06:00:00,01,total,default,718.547415
7,2030-01-01 07:00:00,01,total,default,766.677799
8,2030-01-01 08:00:00,01,total,default,789.203022
9,2030-01-01 09:00:00,01,total,default,801.192525



📊 Value range: 475.56 - 988.30


In [10]:
# Example 4: Segment breakdown (sector analysis)
print("🏢 Query 4: Segment breakdown for national totals 2030")
query4 = build_demand_query(
    start='2030-01-01', end='2031-12-31',
    resolution='1Y', aggregation='sum',
    geography='total', segment='all'
)
print("SQL:")
print(query4)
print("\nResults:")
start_time = time.time()
result4 = db.execute(query4).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")
display(result4)

🏢 Query 4: Segment breakdown for national totals 2030
SQL:

    SELECT
        DATE_TRUNC('year', t.timestamp) AS "period",
        'total' AS geography, t.segment AS segment, 'default' AS scenario_id,
        SUM(t.value) AS value
    FROM (
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                NULL as housing_electrification,
                NULL as transport_electrification, 
                NULL as industry_transition,
                true as is_base
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/base/**/data.parquet', hive_partitioning=FALSE)
         UNION ALL 
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                housing_electrification,
                transport_electrification, 
                industry_transition,
                false as is_base
            FROM parquet_scan('/home/viktor/code/behovskartan/api/data/scenarios/**/data.parquet', hive_partiti

,period,geography,segment,scenario_id,value
0,2030-01-01,total,industry,default,2.051267e+07
1,2030-01-01,total,housing,default,1.138139e+08
2,2030-01-01,total,transport,default,2.370469e+07
3,2031-01-01,total,transport,default,2.411170e+07
4,2031-01-01,total,housing,default,1.157658e+08
5,2031-01-01,total,industry,default,2.086716e+07


## Static Endpoint Generation Queries

These queries generate the data for static API endpoints.

In [ ]:
# Parameters Query - Extract unique parameter values
print("📋 Parameters Query (like /parameters endpoint)")

parameters_query = f"""
WITH data_summary AS (
    SELECT 
        MIN(EXTRACT(year FROM timestamp)) as min_year,
        MAX(EXTRACT(year FROM timestamp)) as max_year,
        array_agg(DISTINCT scenario_id ORDER BY scenario_id) as scenarios,
        array_agg(DISTINCT geography ORDER BY geography) as geographies,
        array_agg(DISTINCT segment ORDER BY segment) as segments
    FROM read_parquet('{parquet_pattern}')
)
SELECT 
    min_year,
    max_year,
    scenarios,
    geographies,
    segments,
    ['1h', '1d', '1w', '1M', '1Y'] as resolutions,
    ['sum', 'mean'] as aggregations
FROM data_summary
"""

start_time = time.time()
parameters_result = db.execute(parameters_query).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")
display(parameters_result)

In [ ]:
# Scenarios Query - List all scenario combinations
print("🎯 Scenarios Query (like /scenarios endpoint)")

scenarios_query = f"""
SELECT 
    scenario_id,
    COUNT(*) as record_count,
    MIN(timestamp) as start_date,
    MAX(timestamp) as end_date,
    CASE WHEN scenario_id = 'default' THEN true ELSE false END as is_default
FROM read_parquet('{parquet_pattern}')
GROUP BY scenario_id
ORDER BY scenario_id
"""

start_time = time.time()
scenarios_result = db.execute(scenarios_query).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")
display(scenarios_result)

In [ ]:
# Geographies Query - Get geography metadata
print("🗺️  Geographies Query (like /geographies endpoint)")

geographies_query = f"""
SELECT 
    geography,
    COUNT(*) as record_count,
    SUM(value) as total_value,
    AVG(value) as avg_value,
    MIN(value) as min_value,
    MAX(value) as max_value
FROM read_parquet('{parquet_pattern}')
WHERE scenario_id = 'default'
GROUP BY geography
ORDER BY geography
"""

start_time = time.time()
geographies_result = db.execute(geographies_query).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")
display(geographies_result.head(10))

## Globals Query - Enhanced Bounds Calculation

This replicates the enhanced globals calculation from `generate-api.js`.

In [ ]:
# Enhanced Globals Query - Multiple bound sets
print("🌍 Enhanced Globals Query (like enhanced /globals endpoint)")

globals_query = f"""
WITH yearly_data AS (
    SELECT
        geography,
        segment,
        DATE_TRUNC('year', timestamp) as year,
        value
    FROM read_parquet('{parquet_pattern}')
    WHERE value IS NOT NULL AND scenario_id = 'default'
),
geography_yearly AS (
    SELECT geography, year, SUM(value) as geography_total
    FROM yearly_data
    GROUP BY geography, year
),
sector_yearly AS (
    SELECT segment, year, SUM(value) as sector_total
    FROM yearly_data
    GROUP BY segment, year
),
national_yearly AS (
    SELECT year, SUM(value) as national_total
    FROM yearly_data
    GROUP BY year
)
SELECT
    -- Raw data bounds
    MIN(yearly_data.value) as raw_min,
    MAX(yearly_data.value) as raw_max,
    COUNT(yearly_data.value) as total_records,
    AVG(yearly_data.value) as mean_value,

    -- Map bounds (geography yearly totals)
    MIN(geography_yearly.geography_total) as map_min,
    MAX(geography_yearly.geography_total) as map_max,

    -- Sector bounds (segment yearly totals)
    MIN(sector_yearly.sector_total) as sector_min,
    MAX(sector_yearly.sector_total) as sector_max,

    -- National bounds (yearly totals)
    MIN(national_yearly.national_total) as national_min,
    MAX(national_yearly.national_total) as national_max

FROM yearly_data
CROSS JOIN geography_yearly
CROSS JOIN sector_yearly
CROSS JOIN national_yearly
"""

print("SQL:")
print(globals_query)
print("\nExecuting...")
start_time = time.time()
globals_result = db.execute(globals_query).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")

# Display results in a more readable format
if len(globals_result) > 0:
    data = globals_result.iloc[0]
    
    print("\n📊 Computed Bounds:")
    print(f"Raw data: {data['raw_min']:.0f} - {data['raw_max']:.0f} Wh ({data['raw_min']/1e6:.1f} - {data['raw_max']/1e6:.1f} MWh)")
    print(f"Map (geography/year): {data['map_min']:.0f} - {data['map_max']:.0f} Wh ({data['map_min']/1e6:.1f} - {data['map_max']/1e6:.1f} MWh)")
    print(f"Sectors (segment/year): {data['sector_min']:.0f} - {data['sector_max']:.0f} Wh ({data['sector_min']/1e9:.1f} - {data['sector_max']/1e9:.1f} GWh)")
    print(f"National (year): {data['national_min']:.0f} - {data['national_max']:.0f} Wh ({data['national_min']/1e12:.1f} - {data['national_max']/1e12:.1f} TWh)")
    print(f"\nTotal records: {int(data['total_records']):,}")
    print(f"Mean value: {data['mean_value']:.2f} Wh")

display(globals_result)

## Validation & Testing Queries

Compare results with API responses and validate data quality.

In [ ]:
# Data Quality Checks
print("🔍 Data Quality Checks")

quality_checks = {
    "Null Values": f"""
        SELECT 
            COUNT(*) as total_records,
            SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) as null_values,
            SUM(CASE WHEN value < 0 THEN 1 ELSE 0 END) as negative_values,
            SUM(CASE WHEN value = 0 THEN 1 ELSE 0 END) as zero_values
        FROM read_parquet('{parquet_pattern}')
    """,
    
    "Partition Completeness": f"""
        SELECT 
            scenario_id,
            COUNT(DISTINCT geography) as geographies,
            COUNT(DISTINCT segment) as segments,
            COUNT(DISTINCT EXTRACT(year FROM timestamp)) as years,
            COUNT(*) as records
        FROM read_parquet('{parquet_pattern}')
        GROUP BY scenario_id
        ORDER BY scenario_id
    """,
    
    "Value Ranges by Segment": f"""
        SELECT 
            segment,
            COUNT(*) as records,
            MIN(value) as min_value,
            MAX(value) as max_value,
            AVG(value) as avg_value,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY value) as median_value
        FROM read_parquet('{parquet_pattern}')
        WHERE scenario_id = 'default'
        GROUP BY segment
        ORDER BY segment
    """
}

for check_name, query in quality_checks.items():
    print(f"\n📋 {check_name}:")
    start_time = time.time()
    result = db.execute(query).df()
    elapsed = time.time() - start_time
    print(f"⏱️  Query time: {elapsed:.3f}s")
    display(result)

In [ ]:
# Performance Benchmarks
print("⚡ Performance Benchmarks")

benchmark_queries = {
    "Simple Aggregation": {
        "query": f"SELECT COUNT(*), SUM(value), AVG(value) FROM read_parquet('{parquet_pattern}')",
        "description": "Basic aggregation across all data"
    },
    
    "Yearly Grouping": {
        "query": f"""
            SELECT 
                EXTRACT(year FROM timestamp) as year,
                SUM(value) as total_value
            FROM read_parquet('{parquet_pattern}')
            WHERE scenario_id = 'default'
            GROUP BY EXTRACT(year FROM timestamp)
            ORDER BY year
        """,
        "description": "Group by year with filtering"
    },
    
    "Complex Multi-dimension": {
        "query": f"""
            SELECT 
                geography,
                segment,
                EXTRACT(year FROM timestamp) as year,
                SUM(value) as total_value
            FROM read_parquet('{parquet_pattern}')
            WHERE scenario_id = 'default'
            GROUP BY geography, segment, EXTRACT(year FROM timestamp)
            ORDER BY geography, segment, year
        """,
        "description": "Multi-dimensional grouping"
    }
}

benchmark_results = []

for name, config in benchmark_queries.items():
    print(f"\n🏃 {name}: {config['description']}")
    
    # Run query multiple times for better timing
    times = []
    for i in range(3):
        start_time = time.time()
        result = db.execute(config['query']).df()
        elapsed = time.time() - start_time
        times.append(elapsed)
    
    avg_time = sum(times) / len(times)
    min_time = min(times)
    max_time = max(times)
    
    print(f"⏱️  Average: {avg_time:.3f}s (min: {min_time:.3f}s, max: {max_time:.3f}s)")
    print(f"📊 Result rows: {len(result)}")
    
    benchmark_results.append({
        'query': name,
        'avg_time': avg_time,
        'min_time': min_time,
        'max_time': max_time,
        'result_rows': len(result)
    })

# Summary table
print("\n📈 Benchmark Summary:")
benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)

## Experimental Queries

Use this section to test new query patterns and explore the data.

In [ ]:
# Experimental Query Area
print("🧪 Experimental Query Area")
print("Use this cell to test new queries and explore the data")

# Example: Time series analysis
experimental_query = f"""
-- Add your experimental queries here
-- Example: Find peak hours across all geographies
SELECT 
    EXTRACT(hour FROM timestamp) as hour_of_day,
    AVG(value) as avg_value,
    COUNT(*) as record_count
FROM read_parquet('{parquet_pattern}')
WHERE scenario_id = 'default' 
  AND EXTRACT(year FROM timestamp) = 2030
GROUP BY EXTRACT(hour FROM timestamp)
ORDER BY hour_of_day
"""

print("Example: Peak hours analysis")
start_time = time.time()
experimental_result = db.execute(experimental_query).df()
elapsed = time.time() - start_time
print(f"⏱️  Query time: {elapsed:.3f}s")
display(experimental_result)

In [ ]:
# Custom Query Helper Function
def run_custom_query(query, description="Custom Query"):
    """
    Helper function to run and time custom queries
    """
    print(f"🔍 {description}")
    print(f"SQL: {query[:100]}..." if len(query) > 100 else f"SQL: {query}")
    
    start_time = time.time()
    try:
        result = db.execute(query).df()
        elapsed = time.time() - start_time
        print(f"⏱️  Query time: {elapsed:.3f}s")
        print(f"📊 Result rows: {len(result)}")
        return result
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"❌ Query failed after {elapsed:.3f}s: {str(e)}")
        return None

# Example usage:
# result = run_custom_query("""
#     SELECT geography, COUNT(*) as records 
#     FROM read_parquet('pattern') 
#     GROUP BY geography
# """, "Geography record counts")

print("✅ Custom query helper function ready")

In [ ]:
# Cleanup
db.close()
print("🧹 Database connection closed")